# Nguyễn Thành Trí - MSSV: 6451071079
## Bài 2 (Trang 28-29) - Câu 6: Biến đổi và mã hóa thuộc tính (Feature Engineering & Encoding)

### Đề bài:
Biến đổi các thuộc tính sau để sẵn sàng cho quá trình khai phá dữ liệu:
1. Chuyển đổi các thuộc tính nhị phân (`default`, `housing`, `loan`, `y`) thành số nguyên (0 và 1).
2. Chia nhóm (Binning) thuộc tính Độ tuổi (`age`) thành 5 nhóm:
   - Nhóm 1: Dưới 25 tuổi
   - Nhóm 2: Từ 25 đến 35 tuổi
   - Nhóm 3: Từ 36 đến 50 tuổi
   - Nhóm 4: Từ 51 đến 60 tuổi
   - Nhóm 5: Trên 60 tuổi
3. Chia nhóm thuộc tính Thời lượng cuộc gọi (`duration`) thành 4 nhóm cân bằng số phần tử bằng `pd.qcut`.
4. Áp dụng mã hóa One-hot encoding cho các thuộc tính phân loại (`marital`, `education`).

In [1]:
# Nguyễn Thành Trí - 6451071079
# Bài 2 (Trang 28-29) - Câu 6: Mã hóa thuộc tính và chia nhóm

import os
import pandas as pd
import numpy as np

print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

def get_data_path(rel_path):
    candidates = [
        rel_path,
        os.path.join("..", rel_path),
        os.path.join("..", "..", rel_path),
        os.path.join("datasets", os.path.basename(rel_path)),
        os.path.join("..", "datasets", os.path.basename(rel_path)),
        os.path.join("..", "..", "datasets", os.path.basename(rel_path)),
        os.path.join("..", "..", "datasets", "Bank", os.path.basename(rel_path)),
        os.path.join("datasets", "Bank", os.path.basename(rel_path))
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Không tìm thấy file dữ liệu: {rel_path}")

file_path = get_data_path("bank.csv")
df = pd.read_csv(file_path, sep=';')

# Điền mode và trích xuất đặc trưng như Câu 5
df['job'] = df['job'].replace('unknown', df[df['job'] != 'unknown']['job'].mode()[0])
df['education'] = df['education'].replace('unknown', df[df['education'] != 'unknown']['education'].mode()[0])
df['contact'] = df['contact'].replace('unknown', df[df['contact'] != 'unknown']['contact'].mode()[0])
df['is_previous_contact'] = (df['pdays'] != -1).astype(int)
df = df.drop(columns=['poutcome', 'pdays'])

# 1. Mã hóa nhị phân yes/no -> 1/0
binary_cols = ['default', 'housing', 'loan', 'y']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0}).astype(int)

# 2. Chia nhóm độ tuổi (age) thành 5 nhóm
age_bins = [0, 25, 35, 50, 60, 100]
age_labels = ['<25', '25-35', '36-50', '51-60', '>60']
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, right=True)

# 3. Chia nhóm thời lượng cuộc gọi (duration) thành 4 nhóm cân bằng (quantiles)
df['duration_group'] = pd.qcut(df['duration'], q=4, labels=['Rất ngắn', 'Ngắn', 'Trung bình', 'Dài'])

# 4. Mã hóa One-hot encoding cho marital và education
df = pd.get_dummies(df, columns=['marital', 'education'], drop_first=True, dtype=int)

print("5 dòng đầu tiên sau khi biến đổi đặc trưng và mã hóa:")
display(df.head())

print()
print("Thống kê phân bố nhóm tuổi (age_group):")
display(df['age_group'].value_counts().sort_index())

print()
print("Thống kê phân bố nhóm thời lượng cuộc gọi (duration_group):")
display(df['duration_group'].value_counts())

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079
5 dòng đầu tiên sau khi biến đổi đặc trưng và mã hóa:


,age,job,default,balance,housing,loan,contact,day,month,duration,campaign,previous,y,is_previous_contact,age_group,duration_group,marital_married,marital_single,education_secondary,education_tertiary
0,30,unemployed,0,1787,0,0,cellular,19,oct,79,1,0,0,0,25-35,Rất ngắn,1,0,0,0
1,33,services,0,4789,1,1,cellular,11,may,220,1,4,0,1,25-35,Trung bình,1,0,1,0
2,35,management,0,1350,1,0,cellular,16,apr,185,1,1,0,1,25-35,Ngắn,0,1,0,1
3,30,management,0,1476,1,1,cellular,3,jun,199,4,0,0,0,25-35,Trung bình,1,0,0,1
4,59,blue-collar,0,0,1,0,cellular,5,may,226,1,0,0,0,51-60,Trung bình,1,0,1,0



Thống kê phân bố nhóm tuổi (age_group):


age_group
<25       111
25-35    1541
36-50    1942
51-60     800
>60       127
Name: count, dtype: int64


Thống kê phân bố nhóm thời lượng cuộc gọi (duration_group):


duration_group
Rất ngắn      1139
Dài           1130
Ngắn          1129
Trung bình    1123
Name: count, dtype: int64

### Nhận xét & Đánh giá (Câu 6):
1. **Mã hóa nhị phân**: Các thuộc tính `default`, `housing`, `loan`, `y` đã được chuyển đổi thành số nguyên [0, 1] chuẩn hóa.
2. **Chia nhóm (Binning)**:
   - Thuộc tính `age` được chia thành 5 khoảng giá trị theo độ tuổi lao động, giúp mô hình hóa dễ dàng hơn so với biến liên tục.
   - Thuộc tính `duration` được chia thành 4 nhóm cân bằng tần số phần tử bằng `pd.qcut` (mỗi nhóm xấp xỉ 25% số lượng bản ghi), khắc phục tình trạng độ lệch lớn (skewness) của thời lượng cuộc gọi.
3. **Mã hóa One-hot**: Các biến phân loại danh nghĩa (`marital`, `education`) được mã hóa thành các cột nhị phân với tham số `drop_first=True` giúp tránh hiện tượng đa cộng tuyến (multicollinearity).